<a href="https://colab.research.google.com/github/Alien-73/Building_Energy_Flex/blob/main/build_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building Simulation

In [ ]:
#Inputs
#----------------------------------------------
#Building Model:

#Type: Residential
#Building model type: 3R2C (Ref: DTU summer course) -> Rie, Rea, Rinf, Ci, Ce
#Parameters
Rea = 58.4895  #['C/kW]
Rie = 1.1521  #['C/kW]
Rinf = 15.748 #['C/kW]
Ce = 4.22     #[kWh/'C]
Ci = 1.15     #[kWh/'C]
Ai = 1.56     #[m^2]
Ae = 0.122    #[m^2]
#----------------------------------------------
#Weather and and External datasets
# reading csv file
df = pd.read_csv("XXX.csv")
#print(df.head())

Qint = df["Qint"]#"Qint":internal process heating [kW]
Qvent = df["Qvent"]#"Qvent":the ventilation heating loss [kW]
Gh = df["Gh"] #"Gh" horizental solar radiation [kW/m.sq]
Ta = df["Ta"] #"Ta" ourdoor temp. [de.C]
#This is unknown #"Q_h":heating input [kW]
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#Weather noise
p11 = -2.89726 #FROM DTU case - Summer course Friday session
p22 = -0.06793 #FROM DTU case - Summer course Friday session
#Noise
Noise_level = 0.5
#generated based on Brownian Motion Noise
np.random.seed(seed=45)
noise_Ti = np.cumsum(np.random.normal(0,sd_Ti , t_eval.size)) #sd_Ti*np.sqrt(dt)
#noise_Th = np.cumsum(np.random.normal(0, sd_Th*np.sqrt(dt), t_eval.size))
noise_Te = np.cumsum(np.random.normal(0,sd_Tm , t_eval.size)) #sd_Te*np.sqrt(dt)

noise_Ti = Noise_level* np.diff(noise_Ti,axis=0)
noise_Te = Noise_level* np.diff(noise_Te,axis=0)
#----------------------------------------
#Simulation Inputs:
dt = 0.15 #Sampling time [hr]
t_sim = 96 #Total Simulation horizon [hr]
Steps_sim = t_sim/dt+1 #Number of steps [-]
#----------------------------------------
#Prices
#Fixed price:
dh_price_sw = 168 * [0.909]
#X = [Ti Te]*T
#U = [Q_h Qvent Qint Ta Gh]*T  #"Qint" "Qvent" "Gh" "Ta" are known
#-------------------------------------
#Heating System
Eff_h = 0.95 #Eff. of the  heating in the building - ATT: should be changed if HP is being used

In [ ]:
#Functions & preprocessing


#Building
#Ordinary Diff. Equation in the following form
#___________________________________________________________________
#dTi/dt = (1/(Ci*Rie)*(Te-Ti) + 1/(Ci*Rinf)*(Ta-Ti) + 1/Ci*(Qh + Qint(x) + Qvent(Already known, I am gonna use them!) + Qdh + Ai*Gh))*dt + dw11*dt
#dTe/dt = (1/(Ce*Rie)*(Ti-Te) + 1/(Ce*Rea)*(Ta-Te) + 1/Ce*(Ae*Gh)*dt + exp(p22)/Cm*dw22)
#___________________________________________________________________

import numpy as np
from scipy.linalg import expm
#Descretsizing the continuous state-space model:
#dx/dt  = A*x + B*u
#y      = C*x + D*u
#to:
#x(k+1) = Ad*x + Bd*u
#y(k)   = C*x + D*u

def discretize(A, B, dT): #A and B as above and T is the timestep (0.25hr)
    # Compute A_d
    A_d = expm(A * dT)

    # Compute B_d using the formula B_d = A_inv * (A_d - I) * B
    # if A is invertible
    n = A.shape[0]
    B_d = np.linalg.inv(A) @ (A_d - np.eye(n)) @ B

    return A_d, B_d

def AB_matrices(Rea, Rie, Rinf, Ci, Ce, eta): #System model
    """
    Build continuous-time A (2x2) and B (2x2) matrices for the 3R2C thermal model:
      [Ti_dot, Te_dot]^T = A [Ti, Te]^T + B [Qh, Qinit, Qvent,T_out, Q_sol]^T
    """
    # Safety valve --> divide-by-zero; small floor
    eps = 1e-9
    Rea  = max(Rea,  eps)
    Rie  = max(Rie,  eps)
    Rinf = max(Rinf, eps)
    Ci   = max(Ci,   eps)
    Ce   = max(Ce,   eps)

    #A (2x2)#
    A = np.array([[-1/(Ci*Rie)-1/(Ci*Rinf), 1/(Ci*Rie)], [1/(Ce*Rie), -1/(Ce*Rie)-1/(Ce*Rea)]])
    #B (2x5)#
    B = np.array([[ 1/Ci, 1/Ci, 1/Ci, 1/(Ci*Rinf) , Ai/Ci], [0 , 0, 0, 1/(Ce*Rea) , Ae/Ce]])
    #A = np.array([[-(1/(Ci*Rie) + 1/(Ci*Rinf)), (1/(Ci*Rie))],
    #             [(1/(Ce*Rie)), -(1/(Ce*Rie) + 1/(Ce*Rea))]])

    #B = np.array([[1/(Ci*Rinf),  eta/Ci],
    #              [1/(Ce*Rea) ,  0.0]])
    return A, B

A, B = AB_matrices(Rea, Rie, Rinf, Ci, Ce, eta)
Ad, Bd = discretize(A, B, dt)

In [ ]:
nx = 2 #The number of states
nu = 1 #The number of controls
#-----------------------------------------

#Cost function weights (?)
Q = casadi.diag([1,0]) #The weights as in Q[0]*(x[0]-x_ref) + Q[1]*(x[1]-x_ref) + ...
R = casadi.diag([0.1,0.5])  #The weights as in R[0]*(u[0]-u_ref) + R[1]*(u[1]-u_ref) + ...
Q_f = casadi.diag([1,1]) #?

#prediction horizon etc. (??)
T = 6     #Number of hr to predict (in my case 12 x 30-min steps which is 6hr)
K = 24     #Prediction horizon (e.g. 24 steps of 15min for me)
dt = 0.25#T/K  #Sample time (here 30 mins)

#constraints
x_lb = [20,-np.inf] #20 #Additon: I can delete these to make the soft constraint on temp.
x_ub = [24,np.inf] #22
u_lb = [0,0]#
u_ub = [6,5]#

#traget values
x_ref = casadi.DM([21,0]) #ADDITION: I only put reference on x_0 (T_in). and T_e we dont care. Change for temp. 21deg.C for example
u_ref = casadi.DM([0,0]) #ADDITION: Keep 0 because we rather not use heat pump to save vost

#Time-series parameters (outdoor temp., solar radiance)
#Generating outdoor temp. signal (I should import the data later)
#t = np.linspace(0, np.pi, 40, endpoint=True)
#T_o = 10*np.sin(t)
#T_o = casadi.DM([0])
##Q_sol = casadi.DM([0])
#Noises on indoor temp. (opeing window, process energy e.i. occupancy change, lamp heat, ...)

total = nx*(K+1) + nu*K #Dimentions of optimization variables
#State_Equations
def make_F(T_o = 0, Q_sol = 0,n_Ti=0,n_Te=0):
  states = casadi.SX.sym("states",nx)
  ctrls = casadi.SX.sym("ctrls",nu) #
  Ti = states[0]
  Te = states[1]
  Qh = ctrls[1]

  #state-space equations - Discrete
  #EQ: X(n+1) = A*X(n) + B*U(n) + Brownian_Motion_noise #,
  [Ti_next,Te_next] = np.dot(A,np.array([Ti,Te])) + np.dot(B,np.array([Qh*Eff_h,T_o,Q_sol])) + np.hstack((n_Ti,n_Te)) #,Qdh*Eff_dh

  states_next = casadi.vertcat(Ti_next,Te_next)
  F = casadi.Function("F",[states,ctrls],[states_next],["x","u"],["x_next"])
  return F